In [21]:
from langchain_community.vectorstores import FAISS
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.documents import Document
from typing_extensions import List, TypedDict
from langchain_classic import hub
from langgraph.graph import StateGraph, START, END
from langchain_core.prompts import PromptTemplate
from langsmith import evaluate, Client
from dotenv import load_dotenv


In [22]:
# load environment variables
load_dotenv()


# load docs from FAISS
embeddings = OpenAIEmbeddings()

persist_path="rag_embeddings"

vector_store = FAISS.load_local(
    folder_path=persist_path,
    embeddings=embeddings,
    allow_dangerous_deserialization=True
    ) 

In [23]:
# langgraph app

llm = ChatOpenAI(model_name = 'gpt-5.4-mini', temperature=0)

prompt = PromptTemplate(
        template="""
        You are an expert chat assistant designed to answer questions based on provided context.
        Carefully use the context below to answer the question. If the context does not contain enough information,
        respond clearly and politely that you do not know the answer.

        Examples:
        Context: The store is located in Manchester, UK on 29th Avenue Street.
        Question: Which city is the store located in?
        Answer: The dtore is located in Manchester.

        Context: No information about dtore location is provided.
        Question: Which city is the store located in?
        Answer: I'm sorry, I don't have enough information to answer that question.

        Context: {context}
        Question: {question}

        Answer:""",
        input_variables=['context', 'question']
    )

# state
class RagState(TypedDict):
    question: str
    context: List[Document]
    answer: str


# nodes
def retrieve(state: RagState):
    retrieved_docs = vector_store.similarity_search(state["question"])
    return {"context": retrieved_docs}


def generate(state: RagState):
    docs_content = "\n\n".join(doc.page_content for doc in state["context"])
    messages = prompt.invoke({"question": state["question"], "context": docs_content})
    response = llm.invoke(messages)
    return {"answer": response.content}

# define graph

graph = StateGraph(RagState)

graph.add_node("retrieve", retrieve)
graph.add_node("generate", generate)

graph.add_edge(START, "retrieve")
graph.add_edge("retrieve", "generate")
graph.add_edge("generate", END)

workflow = graph.compile()

In [ ]:
def run_rag(inputs: dict) -> dict:
    question = inputs.get("Question")
    input_state = {"question": question}
    response = workflow.invoke(input_state)
    return {"Answer": response["answer"]}


In [29]:
client = Client()
dataset_name = "cafe-dataset-from-scratch"


evaluate(
   run_rag,
    # chain.invoke
    # graph.invoke
    data=dataset_name,
    experiment_prefix="rag-cafe-dataset-duplicate experiment 1"
)

Error running target function: 'None' is not an instance of 'str'
Traceback (most recent call last):
  File "c:\Ddrive\Daya\Campusx\langsmith-evals\.venv-1\Lib\site-packages\langsmith\evaluation\_runner.py", line 1949, in _forward
    fn(*args, langsmith_extra=langsmith_extra)
    ~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Ddrive\Daya\Campusx\langsmith-evals\.venv-1\Lib\site-packages\langsmith\run_helpers.py", line 798, in wrapper
    function_result = run_container["context"].run(func, *args, **kwargs)
  File "C:\Users\dayanand.solapure\AppData\Local\Temp\ipykernel_111092\4287026318.py", line 4, in run_rag
    response = workflow.invoke(input_state)
  File "c:\Ddrive\Daya\Campusx\langsmith-evals\.venv-1\Lib\site-packages\langgraph\pregel\main.py", line 4000, in invoke
    for chunk in self.stream(
                 ~~~~~~~~~~~^
        input,
        ^^^^^^
    ...<11 lines>...
        **kwargs,
        ^^^^^^^^^
    ):
    ^
  File "c:\Ddrive\Daya\Campusx\langsmith-evals\.v

View the evaluation results for experiment: 'rag-cafe-dataset-duplicate experiment 1-9258680e' at:
https://smith.langchain.com/o/15cd15b9-c667-4395-8615-69017e774126/datasets/ff712422-3a6b-4e8a-9334-4aa3ab19cb5f/compare?selectedSessions=25f5abd6-3317-4436-9c46-ee96a52a77f2




Error running target function: 'Answer'
Traceback (most recent call last):
  File "c:\Ddrive\Daya\Campusx\langsmith-evals\.venv-1\Lib\site-packages\langsmith\evaluation\_runner.py", line 1949, in _forward
    fn(*args, langsmith_extra=langsmith_extra)
    ~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Ddrive\Daya\Campusx\langsmith-evals\.venv-1\Lib\site-packages\langsmith\run_helpers.py", line 798, in wrapper
    function_result = run_container["context"].run(func, *args, **kwargs)
  File "C:\Users\dayanand.solapure\AppData\Local\Temp\ipykernel_111092\4287026318.py", line 5, in run_rag
    return {"Answer": response["Answer"]}
                      ~~~~~~~~^^^^^^^^^^
KeyError: 'Answer'


<ExperimentResults rag-cafe-dataset-duplicate experiment 1-9258680e>

In [13]:
# Model
evaluate(
    run_rag,
    data=dataset_name,
    experiment_prefix="gpt-3.5-turbo"
)

View the evaluation results for experiment: 'gpt-3.5-turbo-2593bd9f' at:
https://smith.langchain.com/o/ab7cd12a-fb8c-4c9e-89ca-973973263c4f/datasets/a1228fb4-4d67-435a-9f0b-eeff415eb9c0/compare?selectedSessions=c5ade4bd-781d-4417-985b-14bad50a5049




17it [00:15,  1.13it/s]


,inputs.question,outputs.answer,error,reference.output,execution_time,example_id,id
0,Are there any gluten-free options available?,"Yes, there are gluten-free options available s...",None,"Yes, we have gluten-free cookies and a gluten-...",0.788692,2596fdff-a187-4c87-a851-2e7792177723,84ca7ab6-21c6-4f03-aaf4-c90a46af13e4
1,What desserts do you serve at the café?,"The café serves Croissants, Blueberry Muffins,...",None,We have a selection of desserts including chee...,0.764660,fcdc9959-5ae4-4176-973d-47e14faaa035,fedf31ab-5bed-4b05-b4b2-3538d4e9b346
2,What are the café’s specials during autumn and...,The café's specials during autumn are the Seas...,None,"Autumn: Pumpkin Spice Latte, Summer: Fresh Ber...",0.849584,0dcefb7c-757c-4b91-8108-224a44c44d71,6b113544-7cea-4df4-8513-559a253a2069
3,What are the opening hours on weekdays?,The opening hours on weekdays are Monday - Fri...,None,"Monday to Friday, it’s open from 7:30 AM to 8:...",0.736004,18e3c357-ff05-4dd3-9dd8-84b2c2841c43,f234ad48-2fc1-4fd6-bdde-fdde06d1431f
4,Where is Brew & Bite Café located?,"I'm sorry, I don't have enough information to ...",None,"25 Market Street, Manchester, UK.",0.618251,285aba82-3de6-4ffc-af06-a21c3c82b76f,3ef80caa-c836-4d7f-b451-f0c164d480b2
5,Are pizzas available at the café?,"I'm sorry, there is no information provided ab...",None,We don’t sell pizza.,0.866041,33fc353e-e076-4c9d-9ca9-1095af5f2ff7,540d437f-0dec-4639-945f-fbeb316fe62d
6,What can I get under £5 at Brew & Bite Café?,"You can get Iced Coffee, Hot Chocolate, or Her...",None,"Croissants (£2), Blueberry Muffins (£2.20), So...",0.722447,446b8a7b-93ab-4bca-9a9c-e28488585d13,e19c0b97-7bd5-4e49-81d7-0182c0a5d2f2
7,Name three drinks available at the café.,The three drinks available at the café are Ice...,None,"Espresso, Cappuccino, Latte.",1.689729,47789bef-a8e5-411b-b380-4af156cf4839,5f785ea0-816f-411c-92e5-e0b7b91796a0
8,What is the average price range and payment me...,The average price range is £2 - £8 per item. P...,None,£2 – £8 per item. Payment methods include cash...,1.161926,6e18af5f-190f-4e1f-89eb-306701d8c27b,a7d67a30-784c-430a-8dd2-c98c1ed31a04
9,What allergens should I be aware of?,"You should be aware of gluten, dairy, nuts, an...",None,"Contains gluten, dairy, nuts, and soy. Gluten-...",0.615835,6f8d4f71-f790-434a-bed3-11e9f2173c84,47e582b1-5165-4523-9296-4057a4e6e648


In [14]:
# Dataset Split
evaluate(
    run_rag,
    data=client.list_examples(dataset_name=dataset_name, splits=["master"]),  # We pass in a list of Splits
    experiment_prefix="Crucial Examples split"
)

View the evaluation results for experiment: 'Crucial Examples split-8ee08be0' at:
https://smith.langchain.com/o/ab7cd12a-fb8c-4c9e-89ca-973973263c4f/datasets/a1228fb4-4d67-435a-9f0b-eeff415eb9c0/compare?selectedSessions=bffacea0-fadd-400e-a50d-f59b7c1dbae0




5it [00:04,  1.22it/s]


,inputs.question,outputs.answer,error,reference.output,execution_time,example_id,id
0,What are the café’s specials during autumn and...,The café's specials during autumn are the Seas...,None,"Autumn: Pumpkin Spice Latte, Summer: Fresh Ber...",0.786644,0dcefb7c-757c-4b91-8108-224a44c44d71,72647fee-5c26-4a60-b6b4-54c186fc3d95
1,What are the opening hours on weekdays?,The opening hours on weekdays are Monday - Fri...,None,"Monday to Friday, it’s open from 7:30 AM to 8:...",0.680022,18e3c357-ff05-4dd3-9dd8-84b2c2841c43,88b07f4b-87fe-48db-b736-f4213c08032b
2,Where is Brew & Bite Café located?,"I'm sorry, the location of Brew & Bite Café is...",None,"25 Market Street, Manchester, UK.",0.748419,285aba82-3de6-4ffc-af06-a21c3c82b76f,61d5881d-9e69-4c68-9202-74b29e7a071d
3,Are pizzas available at the café?,"I'm sorry, there is no information provided ab...",None,We don’t sell pizza.,0.713119,33fc353e-e076-4c9d-9ca9-1095af5f2ff7,37d06781-2cfb-4cbf-9a38-c855028df5e2
4,What can I get under £5 at Brew & Bite Café?,"You can get Iced Coffee, Hot Chocolate, or Her...",None,"Croissants (£2), Blueberry Muffins (£2.20), So...",0.720765,446b8a7b-93ab-4bca-9a9c-e28488585d13,8780aa36-8c10-4aa9-ae59-2b7db3aaf3b4


In [15]:
# Specific Data Points
evaluate(
    run_rag,
    data=client.list_examples(
        dataset_name=dataset_name, 
        example_ids=[ 
            "6e18af5f-190f-4e1f-89eb-306701d8c27b",
            "7f31f0fa-27c6-45a3-bdd1-1941ce949690"
        ]
    ),
    experiment_prefix="two specific example ids"
)

View the evaluation results for experiment: 'two specific example ids-af9932ea' at:
https://smith.langchain.com/o/ab7cd12a-fb8c-4c9e-89ca-973973263c4f/datasets/a1228fb4-4d67-435a-9f0b-eeff415eb9c0/compare?selectedSessions=bc269ba0-3e5f-4e88-baf9-c801b60473f7




2it [00:02,  1.19s/it]


,inputs.question,outputs.answer,error,reference.output,execution_time,example_id,id
0,What is the average price range and payment me...,The average price range is £2 - £8 per item. P...,None,£2 – £8 per item. Payment methods include cash...,1.131585,6e18af5f-190f-4e1f-89eb-306701d8c27b,a450de6f-0980-4beb-b03e-6c9975454ddb
1,How can I contact the café by phone or email?,You can contact Brew & Bite Café by phone at +...,None,"Phone: +44 161 555 2345, Email: hello@brewandb...",0.774922,7f31f0fa-27c6-45a3-bdd1-1941ce949690,dd92040d-fa64-421f-abfd-d6296cdfa1b4


In [16]:
# Repetitions
evaluate(
    run_rag,
    data=dataset_name,
    experiment_prefix="two repetitions",
    num_repetitions=2   # This field defaults to 1
)

View the evaluation results for experiment: 'two repetitions-33aae754' at:
https://smith.langchain.com/o/ab7cd12a-fb8c-4c9e-89ca-973973263c4f/datasets/a1228fb4-4d67-435a-9f0b-eeff415eb9c0/compare?selectedSessions=816efe05-eb3a-4460-b826-7a9f57fb6870




34it [00:31,  1.08it/s]


,inputs.question,outputs.answer,error,reference.output,execution_time,example_id,id
0,Are there any gluten-free options available?,"Yes, there are gluten-free options available s...",None,"Yes, we have gluten-free cookies and a gluten-...",1.061176,2596fdff-a187-4c87-a851-2e7792177723,c8b9929e-0d93-47b5-8702-29a394b7393c
1,What desserts do you serve at the café?,"The café serves Croissants, Blueberry Muffins,...",None,We have a selection of desserts including chee...,0.915700,fcdc9959-5ae4-4176-973d-47e14faaa035,2c1ec086-ef73-4af5-9c2c-b9624d021fc8
2,What are the café’s specials during autumn and...,The café's specials during autumn are the Seas...,None,"Autumn: Pumpkin Spice Latte, Summer: Fresh Ber...",1.107322,0dcefb7c-757c-4b91-8108-224a44c44d71,c364b922-d349-45a1-ab79-d0cf34142671
3,What are the opening hours on weekdays?,The opening hours on weekdays are Monday - Fri...,None,"Monday to Friday, it’s open from 7:30 AM to 8:...",0.671523,18e3c357-ff05-4dd3-9dd8-84b2c2841c43,f3befdca-05e2-45b1-a7d2-158494691044
4,Where is Brew & Bite Café located?,"I'm sorry, I don't have enough information to ...",None,"25 Market Street, Manchester, UK.",0.847271,285aba82-3de6-4ffc-af06-a21c3c82b76f,65de2abd-70a5-499d-bf5d-58c3bebe135a
5,Are pizzas available at the café?,"I'm sorry, there is no information provided ab...",None,We don’t sell pizza.,0.874951,33fc353e-e076-4c9d-9ca9-1095af5f2ff7,81015fee-969d-4899-a24a-d6cc55620f6d
6,What can I get under £5 at Brew & Bite Café?,"You can get Iced Coffee (£3.80), Hot Chocolate...",None,"Croissants (£2), Blueberry Muffins (£2.20), So...",1.459668,446b8a7b-93ab-4bca-9a9c-e28488585d13,db67526e-6da1-4856-90bb-fbea51891503
7,Name three drinks available at the café.,The three drinks available at the café are Ice...,None,"Espresso, Cappuccino, Latte.",1.202792,47789bef-a8e5-411b-b380-4af156cf4839,fda935ae-cbb3-4d27-a86f-7eb68eed6f40
8,What is the average price range and payment me...,The average price range is £2 - £8 per item. P...,None,£2 – £8 per item. Payment methods include cash...,1.077082,6e18af5f-190f-4e1f-89eb-306701d8c27b,d35a800d-78ae-4305-88bf-c046464899fc
9,What allergens should I be aware of?,"You should be aware of gluten, dairy, nuts, an...",None,"Contains gluten, dairy, nuts, and soy. Gluten-...",0.870679,6f8d4f71-f790-434a-bed3-11e9f2173c84,2fa915c0-dde0-482f-8243-b6e3be4f30a1


In [17]:
evaluate(
    run_rag,
    data=dataset_name,
    experiment_prefix="concurrency",
    max_concurrency=2,  # This defaults to None, so this is an improvement!
)

View the evaluation results for experiment: 'concurrency-84c47b5a' at:
https://smith.langchain.com/o/ab7cd12a-fb8c-4c9e-89ca-973973263c4f/datasets/a1228fb4-4d67-435a-9f0b-eeff415eb9c0/compare?selectedSessions=1d339738-d9f5-40ba-a905-5c63f538473d




17it [00:07,  2.23it/s]


,inputs.question,outputs.answer,error,reference.output,execution_time,example_id,id
0,What desserts do you serve at the café?,"The café serves Croissants, Blueberry Muffins,...",None,We have a selection of desserts including chee...,0.676092,fcdc9959-5ae4-4176-973d-47e14faaa035,4b3434ab-7a00-4e87-b243-67d7dc4f0999
1,Are there any gluten-free options available?,"Yes, there are gluten-free options available s...",None,"Yes, we have gluten-free cookies and a gluten-...",0.896813,2596fdff-a187-4c87-a851-2e7792177723,f076a55a-429e-4eee-90b0-410bf83698ba
2,What are the café’s specials during autumn and...,The café's specials during autumn are the Seas...,None,"Autumn: Pumpkin Spice Latte, Summer: Fresh Ber...",0.837686,0dcefb7c-757c-4b91-8108-224a44c44d71,798a34d7-9dde-4e37-9045-6cb8d41fbbb2
3,What are the opening hours on weekdays?,The opening hours on weekdays are Monday - Fri...,None,"Monday to Friday, it’s open from 7:30 AM to 8:...",1.094857,18e3c357-ff05-4dd3-9dd8-84b2c2841c43,494f6056-536f-4036-8de7-9653ae0520f5
4,Where is Brew & Bite Café located?,"I'm sorry, the location of Brew & Bite Café is...",None,"25 Market Street, Manchester, UK.",0.663756,285aba82-3de6-4ffc-af06-a21c3c82b76f,845017a9-6d53-49fa-a333-db252e74823f
5,Are pizzas available at the café?,"I'm sorry, there is no information provided ab...",None,We don’t sell pizza.,0.754265,33fc353e-e076-4c9d-9ca9-1095af5f2ff7,37d3f623-774c-43c7-8dc8-bd4555877061
6,What can I get under £5 at Brew & Bite Café?,"You can get Iced Coffee, Hot Chocolate, or Her...",None,"Croissants (£2), Blueberry Muffins (£2.20), So...",1.176348,446b8a7b-93ab-4bca-9a9c-e28488585d13,36184054-0829-4c1c-a643-2b3ac5abf0f5
7,Name three drinks available at the café.,The three drinks available at the café are Ice...,None,"Espresso, Cappuccino, Latte.",0.680945,47789bef-a8e5-411b-b380-4af156cf4839,f23b0b4e-ec1f-4baa-bba4-035b1ff419bc
8,What is the average price range and payment me...,The average price range is £2 - £8 per item. P...,None,£2 – £8 per item. Payment methods include cash...,0.704200,6e18af5f-190f-4e1f-89eb-306701d8c27b,cb294d9c-7f5f-48ba-902c-491b7b766072
9,What allergens should I be aware of?,"You should be aware of gluten, dairy, nuts, an...",None,"Contains gluten, dairy, nuts, and soy. Gluten-...",0.753980,6f8d4f71-f790-434a-bed3-11e9f2173c84,155569ba-0281-490b-aa13-4abb615daf2a


In [18]:
# Metadata
evaluate(
    run_rag,
    data=dataset_name,
    experiment_prefix="metadata added",
    metadata={ "model_name": 'xyz'
    }
)

View the evaluation results for experiment: 'metadata added-32e9abdc' at:
https://smith.langchain.com/o/ab7cd12a-fb8c-4c9e-89ca-973973263c4f/datasets/a1228fb4-4d67-435a-9f0b-eeff415eb9c0/compare?selectedSessions=41883094-e3e3-477b-a02e-09e2b1b4a7c6




17it [00:16,  1.03it/s]


,inputs.question,outputs.answer,error,reference.output,execution_time,example_id,id
0,Are there any gluten-free options available?,"Yes, there are gluten-free options available s...",None,"Yes, we have gluten-free cookies and a gluten-...",1.085535,2596fdff-a187-4c87-a851-2e7792177723,2e9d5007-a009-4316-9a07-38128d8cf54a
1,What desserts do you serve at the café?,"The café serves Croissants, Blueberry Muffins,...",None,We have a selection of desserts including chee...,1.357887,fcdc9959-5ae4-4176-973d-47e14faaa035,8b7d6ae2-33b3-4ac1-a000-64184b30e988
2,What are the café’s specials during autumn and...,The café's specials during autumn are the Seas...,None,"Autumn: Pumpkin Spice Latte, Summer: Fresh Ber...",0.820693,0dcefb7c-757c-4b91-8108-224a44c44d71,9e244e16-6492-4429-8fc6-31579c6aca31
3,What are the opening hours on weekdays?,The opening hours on weekdays are Monday - Fri...,None,"Monday to Friday, it’s open from 7:30 AM to 8:...",0.900306,18e3c357-ff05-4dd3-9dd8-84b2c2841c43,f5853251-493e-410b-8a50-fe6330ed4ba0
4,Where is Brew & Bite Café located?,"I'm sorry, the location of Brew & Bite Café is...",None,"25 Market Street, Manchester, UK.",0.644890,285aba82-3de6-4ffc-af06-a21c3c82b76f,3db45566-3ee1-48d7-a3e7-c983ddb415a6
5,Are pizzas available at the café?,"I'm sorry, there is no information provided ab...",None,We don’t sell pizza.,0.736996,33fc353e-e076-4c9d-9ca9-1095af5f2ff7,991fd518-b452-4f90-9d82-14b1608b1f23
6,What can I get under £5 at Brew & Bite Café?,"You can get Iced Coffee, Hot Chocolate, or Her...",None,"Croissants (£2), Blueberry Muffins (£2.20), So...",0.784871,446b8a7b-93ab-4bca-9a9c-e28488585d13,b2fb05bb-bb1a-4e4e-a325-881941ab6615
7,Name three drinks available at the café.,The three drinks available at the café are Ice...,None,"Espresso, Cappuccino, Latte.",0.764218,47789bef-a8e5-411b-b380-4af156cf4839,2744b6d2-7404-4357-ad43-e55c172bbc2d
8,What is the average price range and payment me...,The average price range is £2 - £8 per item. P...,None,£2 – £8 per item. Payment methods include cash...,0.799072,6e18af5f-190f-4e1f-89eb-306701d8c27b,b58c780f-b7f1-475e-bd84-13d17fb7b1dc
9,What allergens should I be aware of?,"You should be aware of gluten, dairy, nuts, an...",None,"Contains gluten, dairy, nuts, and soy. Gluten-...",0.621631,6f8d4f71-f790-434a-bed3-11e9f2173c84,3f0b4381-8108-4281-b927-23ca3c62a160
